Common tables used across analyses, we save them to the bucket here so that they're downloaded only once.

In [ ]:
# Common imports and constants
from pathlib import Path
from pandas_gbq import read_gbq
import pandas as pd

bucket_mount_point = Path.home() / 'workspace/scalable-vus-validation'

demographics_path = bucket_mount_point / 'cohorts/demo.tsv.gz'
ancestry_pca_path = bucket_mount_point / 'cohorts/ancestry_pca.tsv.gz'

ancestry_preds_path = Path.home() / 'workspace/vwb-aou-datasets-controlled-v8r9/v8/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv'
dataset = 'wb-silky-artichoke-2408.C2024Q3R9'
use_bqstorage_api = True

In [ ]:
(bucket_mount_point / 'cohorts').mkdir(parents=True, exist_ok=True)

## Demographics and event count table for all WGS participants

In [ ]:
# Get event count statistics

count_events_sql = f"""
    SELECT *
    FROM (
        SELECT
            person_id,
            COUNT(DISTINCT condition_start_date) AS n_events,
            MIN(condition_start_date) AS first_event,
            MAX(condition_start_date) AS last_event
        FROM `{dataset}.condition_occurrence`
        WHERE
            person_id IN (
                SELECT person_id FROM `{dataset}.cb_search_person`
                WHERE has_whole_genome_variant = 1
            )
        GROUP BY person_id
    ) INNER JOIN (
        SELECT
            person_id,
            sex_at_birth,
            dob,
            age_at_cdr
        FROM `{dataset}.cb_search_person`
    ) USING (person_id)
"""

count_events_df = read_gbq(
    count_events_sql,
    index_col='person_id',
    dialect='standard',
    use_bqstorage_api=use_bqstorage_api,
    progress_bar_type='tqdm_notebook'
).assign(
    first_event = lambda df: pd.to_datetime(df.first_event),
    last_event = lambda df: pd.to_datetime(df.last_event),
    dob = lambda df: pd.to_datetime(df.dob)
)

In [ ]:
count_events_df

In [ ]:
count_events_df.to_csv(demographics_path, sep='\t')

## Ancestry PCA table

In [ ]:
ancestry_df = pd.read_table(ancestry_preds_path)

In [ ]:
# Ancestry PCA
from ast import literal_eval

pca_df = pd.concat(
    [
        ancestry_df[['research_id']],
        ancestry_df.pca_features.apply(lambda x: pd.Series(literal_eval(x))).rename(columns=lambda c: f'pca_{c}')
    ],
    axis='columns'
)

In [ ]:
pca_df

In [ ]:
pca_df.to_csv(ancestry_pca_path, sep='\t', index=False)